# 06 - Predictive Analytics: Wind Prediction

Can we predict tomorrow's wind from today's data?

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S
from ocean_data_workshop.dsn import dsn
import psycopg

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

# Setup
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ML Workshop Environment Ready")
print("=" * 60)
print("Data sources:")
print("  - NDBC 46092 (wind)")
print("  - GLORYS12V1 (ocean)")
print("  - SanctSound MB01 (acoustic)")
print("=" * 60)

ML Workshop Environment Ready
Data sources:
  - NDBC 46092 (wind)
  - GLORYS12V1 (ocean)
  - SanctSound MB01 (acoustic)


## The problem

Predict wind speed for day T+1 using data from day T and ocean conditions.

This is a regression problem - predict a continuous value.

In [2]:
import numpy as np
import pandas as pd

# Load wind data
conn = psycopg.connect(dsn())

query = "SELECT observed_at, station_id, wind_speed_mean_ms, wind_gust_max_ms,"
query += " wind_direction_deg, wind_northward_ms, wave_height_max_m,"
query += " air_temp_mean_c, water_temp_mean_c"
query += " FROM wind_daily WHERE station_id = '46092' ORDER BY observed_at"

wind_df = pd.read_sql_query(query, conn)
conn.close()

print(f"Wind data: {len(wind_df):,} daily measurements")
print(f"Date range: {wind_df['observed_at'].min()} to {wind_df['observed_at'].max()}")

# Create target: next day's wind speed
wind_df['target_wind'] = wind_df['wind_speed_mean_ms'].shift(-1)

# Drop last row (no target for last day)
wind_df = wind_df.dropna(subset=['target_wind'])

print(f"\nDataset with target: {len(wind_df):,} rows")

# Features: current day wind + date features
wind_df['month'] = wind_df['observed_at'].dt.month
wind_df['dayofyear'] = wind_df['observed_at'].dt.dayofyear
wind_df['cos_month'] = np.cos(2 * np.pi * wind_df['month'] / 12)
wind_df['sin_month'] = np.sin(2 * np.pi * wind_df['month'] / 12)

# Define features
feature_cols = ['wind_speed_mean_ms', 'wind_gust_max_ms', 'wind_direction_deg',
                'wind_northward_ms', 'wave_height_max_m', 'air_temp_mean_c',
                'water_temp_mean_c', 'cos_month', 'sin_month']

X = wind_df[feature_cols].values
y = wind_df['target_wind'].values

print(f"\nFeatures ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  - {f}")
print("Target: target_wind (tomorrow's wind speed)")

# Split
split_idx = int(len(X) * 0.75)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"\nTrain: {len(y_train)}, Test: {len(y_test)}")

Wind data: 784 daily measurements
Date range: 2019-01-05 00:00:00+00:00 to 2021-05-01 00:00:00+00:00

Dataset with target: 783 rows

Features (9):
  - wind_speed_mean_ms
  - wind_gust_max_ms
  - wind_direction_deg
  - wind_northward_ms
  - wave_height_max_m
  - air_temp_mean_c
  - water_temp_mean_c
  - cos_month
  - sin_month
Target: target_wind (tomorrow's wind speed)

Train: 587, Test: 196


/var/folders/6_/6bnkvg2j2qd8720cj2rr98080000gn/T/ipykernel_24734/841662593.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  wind_df = pd.read_sql_query(query, conn)


## Baseline models

What's a good baseline for wind prediction?

In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Baseline 1: Always predict the mean
mean_wind = y_train.mean()
y_pred_mean = np.full_like(y_test, mean_wind)
mae_mean = mean_absolute_error(y_test, y_pred_mean)
print(f"Baseline (mean): MAE = {mae_mean:.2f} m/s")

# Baseline 2: Persistence (tomorrow = today)
# Shift wind speed by 1 day
wind_df['persistence'] = wind_df['wind_speed_mean_ms'].shift(1)
wind_df = wind_df.dropna(subset=['persistence'])

# Align with target
idx = wind_df.index
split_idx2 = int(len(idx) * 0.75)
train_idx = idx[:split_idx2]
test_idx = idx[split_idx2:]

# Get persistence predictions for test set
persist_df = wind_df.loc[test_idx]
y_pred_persist = persist_df['persistence'].values
y_test_persist = persist_df['target_wind'].values

mae_persist = mean_absolute_error(y_test_persist, y_pred_persist)
print(f"Baseline (persistence): MAE = {mae_persist:.2f} m/s")

print("\nBaseline comparison:")
print(f"  Mean baseline:     {mae_mean:.2f} m/s")
print(f"  Persistence:       {mae_persist:.2f} m/s")
if mae_persist < mae_mean:
    print("  Better: Persistence")
else:
    print("  Better: Mean")

Baseline (mean): MAE = 1.39 m/s
Baseline (persistence): MAE = 1.44 m/s

Baseline comparison:
  Mean baseline:     1.39 m/s
  Persistence:       1.44 m/s
  Better: Mean


## Train a real model

Let's train a Random Forest regressor.

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

# Predict
y_pred_rf = rf.predict(X_test_scaled)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print(f"  MAE:  {mae_rf:.2f} m/s")
print(f"  R2:   {r2_rf:.3f}")

# Compare to baseline
print("\nImprovement over baseline:")
print(f"  Baseline MAE:  {mae_persist:.2f} m/s")
print(f"  Model MAE:     {mae_rf:.2f} m/s")
if mae_persist > mae_rf:
    improvement = 100 * (mae_persist - mae_rf) / mae_persist
    print(f"  Improvement:   {mae_persist - mae_rf:.2f} m/s ({improvement:.1f}%)")
else:
    print("  Model did worse than baseline")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 features:")
print(feature_importance.head())

Random Forest Results:
  MAE:  1.24 m/s
  R2:   0.111

Improvement over baseline:
  Baseline MAE:  1.44 m/s
  Model MAE:     1.24 m/s
  Improvement:   0.20 m/s (14.2%)

Top 5 features:
              feature  importance
0  wind_speed_mean_ms    0.386261
6   water_temp_mean_c    0.236538
3   wind_northward_ms    0.111632
2  wind_direction_deg    0.100574
5     air_temp_mean_c    0.095698


/Users/dev/Developer/projects/ocean-sim/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:1211: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/dev/Developer/projects/ocean-sim/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:1216: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/dev/Developer/projects/ocean-sim/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:1240: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


## The lesson

- Wind prediction is challenging (natural variability)
- Persistence baseline is strong (weather is autocorrelated)
- Random Forest can outperform baselines
- Feature importance tells us what matters